In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from IPython import get_ipython

tl.set_backend("pytorch")

torch.manual_seed(42)
np.random.seed(42)

GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports (with fallback support)
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions, ActivationHookManager
    from neural_decomp.decomposition import (
        decompose_svd,
        compute_spectral_energy,
        truncate_svd,
        decompose_tucker,
    )
    from neural_decomp.profiling import (
        summarize_activations,
        partition_inactive_neurons,
        compute_dead_neuron_ratio,
        compute_outlier_ratio,
    )
    from neural_decomp.evaluation import evaluate_mnli
    from neural_decomp.utils import get_device_info, save_json_metrics, print_section
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions, ActivationHookManager
    from utility.decomposition import (
        decompose_svd,
        compute_spectral_energy,
        truncate_svd,
        decompose_tucker,
    )
    from utility.profiling import (
        summarize_activations,
        partition_inactive_neurons,
        compute_dead_neuron_ratio,
        compute_outlier_ratio,
    )
    from utility.evaluation import evaluate_mnli
    from utility.utils import get_device_info, save_json_metrics, print_section
    print("Loaded utility library.")

# Inspect hardware environment
device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")
if device_info["cuda_available"]:
    print(f"Allocated VRAM: {device_info.get('allocated_gb', 0.0)} GB")

In [ ]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer
# =====================================================================
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.BALANCED,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

# Target Submodule: Layer 0 MLP gate projection
target_layer = model.model.layers[0]
target_submodule = target_layer.mlp.gate_proj

print(f"Model ID: {model_id}")
print(f"Target Layer: Layer 0 ({target_layer.__class__.__name__})")
print(f"Target Submodule: gate_proj ({target_submodule.weight.shape[0]} x {target_submodule.weight.shape[1]})")
print(f"Total Submodule Parameters: {target_submodule.weight.numel():,}")

# Keep an immutable reference copy of pristine original weights for clean rollbacks
W_gate_orig = target_submodule.weight.data.clone()

In [ ]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

# Configure evaluation sample size for interactive experiments
EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI validation split: {len(ds):,} total samples")
print(f"Active Evaluation Subset: {len(eval_data):,} samples")
print(f"Candidate Label IDs: {dict(zip(label_names, label_token_ids))}")

In [ ]:
# =====================================================================
# STEP 4: Capture Layer 0 Activations & Establish Baseline Accuracy
# =====================================================================
hook_mgr = ActivationHookManager(target_layer)

benchmark_activations = {
    name: [] for name, _ in target_layer.named_modules() if name != ""
}
current_step_acts = {}

def capture_hook(submodule_name):
    def hook(module, input, output):
        act = output[0] if isinstance(output, tuple) else output
        current_step_acts[submodule_name] = act.detach().cpu()
    return hook

handles = []
for name, submodule in target_layer.named_modules():
    if name != "":
        h = submodule.register_forward_hook(capture_hook(name))
        handles.append(h)

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Inference & Hook Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        # Collect activations for each submodule
        for name, act_tensor in current_step_acts.items():
            sample_summary = act_tensor.squeeze(0).mean(dim=0)
            benchmark_activations[name].append(sample_summary)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

# Remove hook handles
for h in handles:
    h.remove()

baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print("Classification Report:")
print(classification_report(ground_truth, predictions, labels=[0, 1, 2], target_names=label_names, zero_division=0))

In [ ]:
# =====================================================================
# STEP 5: Pool & Format Activation Profiles
# =====================================================================
for name in benchmark_activations:
    pooled_samples = []
    for item in benchmark_activations[name]:
        arr = item.cpu().numpy() if isinstance(item, torch.Tensor) else np.array(item)
        if arr.ndim == 2:
            pooled = arr.mean(axis=0)
        elif arr.ndim == 3:
            pooled = arr.mean(axis=1).reshape(-1)
        else:
            pooled = arr
        pooled_samples.append(pooled)
    benchmark_activations[name] = np.stack(pooled_samples)

act_fn_acts = benchmark_activations["mlp.act_fn"]  # Shape: [num_samples, 6912]
print(f"Extracted 'mlp.act_fn' activation matrix shape: {act_fn_acts.shape}")

# Summarize distribution metrics
act_summary = summarize_activations(act_fn_acts)
print(f"MLP act_fn Mean:         {act_summary['mean']:.4f}")
print(f"MLP act_fn Std Dev:      {act_summary['std']:.4f}")
print(f"MLP act_fn Dead (<0.05): {act_summary['dead_ratio_pct']:.2f}%")
print(f"MLP act_fn Outliers (>3): {act_summary['outlier_ratio_pct']:.2f}%")

In [ ]:
# =====================================================================
# STEP 6: Partition Neurons into Active vs Inactive Subsets
# =====================================================================
NUM_ACTIVE = 2400  # Highly factorable: 2400 = 40 * 60 or 24 * 100
total_neurons = act_fn_acts.shape[1]
num_inactive = total_neurons - NUM_ACTIVE

neuron_variances = np.var(act_fn_acts, axis=0)
sorted_neuron_indices = np.argsort(neuron_variances)

# Inactive are lowest variance; Active are highest variance
inactive_indices = sorted_neuron_indices[:num_inactive]
active_indices = sorted_neuron_indices[num_inactive:]

print(f"Total Neurons:     {total_neurons:,}")
print(f"Active Neurons:    {len(active_indices):,} ({len(active_indices)/total_neurons*100:.1f}%)")
print(f"Inactive Neurons:  {len(inactive_indices):,} ({len(inactive_indices)/total_neurons*100:.1f}%)")
print(f"Active Variance Range:   [{neuron_variances[active_indices].min():.5f}, {neuron_variances[active_indices].max():.5f}]")
print(f"Inactive Variance Range: [{neuron_variances[inactive_indices].min():.5f}, {neuron_variances[inactive_indices].max():.5f}]")

# Extract the weight slices
W_active = W_gate_orig[active_indices, :].float().cpu()
W_inactive = W_gate_orig[inactive_indices, :].float().cpu()

print(f"\nW_active shape:   {list(W_active.shape)} ({W_active.numel():,} parameters)")
print(f"W_inactive shape: {list(W_inactive.shape)} ({W_inactive.numel():,} parameters)")

# Plot Activation Variance Distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(neuron_variances[inactive_indices], bins=40, color='gray', alpha=0.7, label='Inactive Neurons')
ax[0].hist(neuron_variances[active_indices], bins=40, color='crimson', alpha=0.7, label='Active Neurons')
ax[0].set_title("Neuron Activation Variance Distribution")
ax[0].set_xlabel("Activation Variance")
ax[0].set_ylabel("Neuron Count")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

sorted_vars = np.sort(neuron_variances)
ax[1].plot(sorted_vars, color='darkblue', linewidth=2)
ax[1].axvline(num_inactive, color='crimson', linestyle='--', label=f'Active Partition Cutoff ({num_inactive})')
ax[1].set_title("Sorted Neuron Variance Curve")
ax[1].set_xlabel("Neuron Rank (Ascending Activity)")
ax[1].set_ylabel("Variance")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# STEP 7: Compute SVD & Contrast Spectral Decay Curves
# =====================================================================
U_act, S_act, Vh_act = decompose_svd(W_active)
U_inact, S_inact, Vh_inact = decompose_svd(W_inactive)

energy_act = compute_spectral_energy(S_act)
energy_inact = compute_spectral_energy(S_inact)

total_dims = len(S_act)  # min(2400, 1152) = 1152

print("Spectral Energy Retention Comparison (Active vs Inactive):")
print(f"Threshold | Active Rank Needed (out of {total_dims}) | Inactive Rank Needed")
print(f"  80%     | {energy_act['rank_80']:<4} ({energy_act['rank_80']/total_dims*100:.1f}%)               | {energy_inact['rank_80']:<4} ({energy_inact['rank_80']/total_dims*100:.1f}%)")
print(f"  90%     | {energy_act['rank_90']:<4} ({energy_act['rank_90']/total_dims*100:.1f}%)               | {energy_inact['rank_90']:<4} ({energy_inact['rank_90']/total_dims*100:.1f}%)")
print(f"  95%     | {energy_act['rank_95']:<4} ({energy_act['rank_95']/total_dims*100:.1f}%)               | {energy_inact['rank_95']:<4} ({energy_inact['rank_95']/total_dims*100:.1f}%)")
print(f"  99%     | {energy_act['rank_99']:<4} ({energy_act['rank_99']/total_dims*100:.1f}%)               | {energy_inact['rank_99']:<4} ({energy_inact['rank_99']/total_dims*100:.1f}%)")

# Plot singular values and cumulative energy
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(S_act.numpy(), label='Active Weights (W_active)', color='crimson', linewidth=2)
ax[0].plot(S_inact.numpy(), label='Inactive Weights (W_inactive)', color='royalblue', linewidth=2)
ax[0].set_title("Singular Value Spectra Comparison")
ax[0].set_xlabel("Singular Value Rank")
ax[0].set_ylabel("Singular Value (σ)")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

cum_energy_act = (torch.cumsum(S_act**2, dim=0) / torch.sum(S_act**2)).numpy()
cum_energy_inact = (torch.cumsum(S_inact**2, dim=0) / torch.sum(S_inact**2)).numpy()

ax[1].plot(cum_energy_act, label='Active Weights Energy', color='crimson', linewidth=2)
ax[1].plot(cum_energy_inact, label='Inactive Weights Energy', color='royalblue', linewidth=2)
ax[1].axhline(0.95, color='gray', linestyle='--', label='95% Energy Threshold')
ax[1].axvline(energy_act['rank_95'], color='crimson', linestyle=':', label=f"Active 95% ({energy_act['rank_95']})")
ax[1].axvline(energy_inact['rank_95'], color='royalblue', linestyle=':', label=f"Inactive 95% ({energy_inact['rank_95']})")
ax[1].set_title("Cumulative Spectral Energy Retention")
ax[1].set_xlabel("Rank")
ax[1].set_ylabel("Fraction of Total Energy")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# STEP 8: Strategy 1 — Low-Rank SVD Reconstruction & Evaluation
# =====================================================================
svd_test_configs = [
    {"label": "SVD (99% Energy)", "rank": energy_act["rank_99"]},
    {"label": "SVD (95% Energy)", "rank": energy_act["rank_95"]},
    {"label": "SVD (90% Energy)", "rank": energy_act["rank_90"]},
]

svd_results = []

for cfg in svd_test_configs:
    r = cfg["rank"]
    # Reconstruct 2D slice
    W_act_svd = truncate_svd(U_act, S_act, Vh_act, rank=r)
    
    # Relative Frobenius reconstruction error
    rel_error = (torch.norm(W_active - W_act_svd) / torch.norm(W_active)).item()
    
    # Parameter accounting for factorized representation: U_r (M x r) + S_r (r) + V_r (r x N)
    m, n = W_active.shape
    factorized_params = (m * r) + r + (r * n)
    orig_params = m * n
    compression_ratio = orig_params / factorized_params
    eliminated_pct = (1.0 - factorized_params / orig_params) * 100.0
    
    print(f"\nEvaluating {cfg['label']} (Rank {r} / {total_dims})...")
    print(f"  Reconstruction Error: {rel_error * 100:.2f}%")
    print(f"  Factorized Parameters: {factorized_params:,} vs Original {orig_params:,} ({compression_ratio:.2f}x)")
    
    # Inject reconstructed weights into live model
    target_submodule.weight.data = W_gate_orig.clone()
    target_submodule.weight.data[active_indices, :] = W_act_svd.to(
        device=model.device, dtype=target_submodule.weight.dtype
    )
    
    # Evaluate live accuracy on MNLI
    predictions = []
    ground_truth = []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {cfg['label']}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]
            pred_label = torch.argmax(next_token_logits[label_token_ids]).item()
            predictions.append(pred_label)
            ground_truth.append(sample["label"])
            
    acc = accuracy_score(ground_truth, predictions)
    delta = acc - baseline_accuracy
    print(f"  Final Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    
    svd_results.append({
        "strategy": cfg["label"],
        "rank": r,
        "parameters": factorized_params,
        "compression_ratio": compression_ratio,
        "eliminated_pct": eliminated_pct,
        "recon_error_pct": rel_error * 100.0,
        "accuracy": acc,
        "delta": delta,
    })

# Restore baseline weights
target_submodule.weight.data = W_gate_orig.clone()
print("\nBaseline weights restored.")

In [ ]:
# =====================================================================
# STEP 9: Strategy 2 — Co-Activation Cluster-Guided 4D Tucker Decomposition
# =====================================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

# 1. Cluster active neurons by activation profile correlation
active_profiles = act_fn_acts[:, active_indices].T  # [2400, num_samples]
active_norm = normalize(active_profiles, norm='l2', axis=1)

num_clusters = 6
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(active_norm)

# 2. Permute active weights by cluster so correlated neurons are contiguous
cluster_order = np.argsort(cluster_labels)
W_active_clustered = W_active[cluster_order, :]

# 3. Fold into 4D tensor with contiguous cluster structure
tensor_shape = [40, 60, 24, 48]
assert math.prod(tensor_shape) == W_active.numel(), "Tensor dimensions must match weight elements!"
W_clustered_tensor = W_active_clustered.reshape(*tensor_shape).float().cpu()
print(f"Folded cluster-aligned weights into 4D Tensor: {list(W_clustered_tensor.shape)}")

tucker_configs = [
    {"label": "Cluster-Tucker (High-Capacity)", "ranks": [32, 48, 18, 36]},
    {"label": "Cluster-Tucker (Moderate)",      "ranks": [26, 38, 14, 28]},
    {"label": "Cluster-Tucker (Aggressive)",    "ranks": [20, 30, 12, 24]},
]

tucker_results = []

for cfg in tucker_configs:
    ranks = cfg["ranks"]
    core, factors, recon_tensor, metrics = decompose_tucker(W_clustered_tensor, rank=ranks, init='svd')
    
    # Unfold reconstructed tensor and invert cluster permutation back to original index order
    W_recon_clustered = recon_tensor.reshape(NUM_ACTIVE, 1152)
    inv_order = np.argsort(cluster_order)
    W_act_tucker = W_recon_clustered[inv_order, :]
    
    # Calculate error
    rel_error = (torch.norm(W_active - W_act_tucker) / torch.norm(W_active)).item()
    
    print(f"\nEvaluating {cfg['label']} (Ranks {ranks})...")
    print(f"  Reconstruction Error: {rel_error * 100:.2f}%")
    print(f"  Core Parameters:      {metrics['core_params']:,}")
    print(f"  Factor Parameters:    {metrics['factors_params']:,}")
    print(f"  Total Compressed:     {metrics['total_compressed_params']:,} ({metrics['compression_ratio']:.2f}x)")
    print(f"  Parameters Cut:       {metrics['eliminated_pct']:.2f}%")
    
    # Inject into live model
    target_submodule.weight.data = W_gate_orig.clone()
    target_submodule.weight.data[active_indices, :] = W_act_tucker.to(
        device=model.device, dtype=target_submodule.weight.dtype
    )
    
    # Evaluate live accuracy on MNLI
    predictions = []
    ground_truth = []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {cfg['label']}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]
            pred_label = torch.argmax(next_token_logits[label_token_ids]).item()
            predictions.append(pred_label)
            ground_truth.append(sample["label"])
            
    acc = accuracy_score(ground_truth, predictions)
    delta = acc - baseline_accuracy
    print(f"  Final Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    
    tucker_results.append({
        "strategy": cfg["label"],
        "ranks": ranks,
        "parameters": metrics["total_compressed_params"],
        "compression_ratio": metrics["compression_ratio"],
        "eliminated_pct": metrics["eliminated_pct"],
        "recon_error_pct": rel_error * 100.0,
        "accuracy": acc,
        "delta": delta,
    })

# Restore baseline weights
target_submodule.weight.data = W_gate_orig.clone()
print("\nBaseline weights restored.")

In [ ]:
# =====================================================================
# STEP 10: Comparative Analysis, Global Time Log & Results Export
# =====================================================================
all_entries = [
    {
        "Variant": "Baseline (Uncompressed)",
        "Parameters": W_active.numel(),
        "Ratio": "1.00x",
        "Reduction (%)": "0.0%",
        "Recon Err (%)": "0.0%",
        "Accuracy (%)": f"{baseline_accuracy * 100:.2f}%",
        "Delta vs Base": "+0.00%",
    }
]

for res in svd_results + tucker_results:
    all_entries.append({
        "Variant": res["strategy"],
        "Parameters": int(res["parameters"]),
        "Ratio": f"{res['compression_ratio']:.2f}x",
        "Reduction (%)": f"{res['eliminated_pct']:.1f}%",
        "Recon Err (%)": f"{res['recon_error_pct']:.2f}%",
        "Accuracy (%)": f"{res['accuracy'] * 100:.2f}%",
        "Delta vs Base": f"{res['delta'] * 100:+.2f}%",
    })

print("=" * 95)
print(f"{'Variant':<28} | {'Params':<9} | {'Ratio':<6} | {'Cut %':<6} | {'Err %':<7} | {'Accuracy':<9} | {'Delta':<7}")
print("=" * 95)
for row in all_entries:
    print(f"{row['Variant']:<28} | {row['Parameters']:<9} | {row['Ratio']:<6} | {row['Reduction (%)']:<6} | {row['Recon Err (%)']:<7} | {row['Accuracy (%)']:<9} | {row['Delta vs Base']:<7}")
print("=" * 95)

# Global Notebook Timing
total_notebook_runtime = time.time() - GLOBAL_NOTEBOOK_START_TIME
print(f"cummulative_time: {total_notebook_runtime:.2f}s")

# Save experiment run to artifacts
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
output_path = artifact_dir / "04_active_acts_non_naive_results.json"

export_data = {
    "model_id": model_id,
    "eval_samples": len(eval_data),
    "active_neurons": NUM_ACTIVE,
    "baseline_accuracy": baseline_accuracy,
    "results": all_entries,
    "timing_summary": {
        "cummulative_time_sec": round(total_notebook_runtime, 3),
        "cell_timings": NOTEBOOK_TIMINGS,
    },
}

save_json_metrics(export_data, output_path)
print(f"\nBenchmark results & timing data saved to: {output_path}")

# Plot Accuracy vs Parameters
variants = [row['Variant'] for row in all_entries]
accs = [float(row['Accuracy (%)'].replace('%', '')) for row in all_entries]
errors = [float(row['Recon Err (%)'].replace('%', '')) for row in all_entries]

fig, ax1 = plt.subplots(figsize=(12, 5))
x = np.arange(len(variants))

color = 'tab:blue'
ax1.set_xlabel('Compression Strategy', fontweight='bold')
ax1.set_ylabel('MNLI Accuracy (%)', color=color, fontweight='bold')
bars = ax1.bar(x - 0.15, accs, width=0.3, color=color, label='Accuracy (%)', alpha=0.85)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(x)
ax1.set_xticklabels(variants, rotation=25, ha='right')
ax1.set_ylim(bottom=max(0, min(accs) - 10), top=max(accs) + 5)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Weight Reconstruction Error (%)', color=color, fontweight='bold')
lines = ax2.plot(x, errors, color=color, marker='o', linewidth=2, label='Recon Error (%)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title("Active Slice Compression: Accuracy Retention vs Weight Error", fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()